# Feature Selection

In [1]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

os.chdir("/Users/julianguyen/Documents/multimodal-pmlb")

np.random.seed(101)

### Load in matrices

In [2]:
atc = pd.read_csv("data/procdata/files/atac.csv", index_col=0).T
rna = pd.read_csv("data/procdata/files/rna.csv", index_col=0).T
cnv = pd.read_csv("data/procdata/files/cnv.csv", index_col=0).T
mut = pd.read_csv("data/procdata/files/mut.csv", index_col=0).T

# load in targets
targets = pd.read_csv("data/procdata/files/meta.csv").set_index("PMLB_organoidID").loc[:, ["doubling_rate", "primary_tumor_site"]]
dt = targets["doubling_rate"]

# get indices
assert atc.index.equals(rna.index)
assert atc.index.equals(cnv.index)
assert atc.index.equals(mut.index)
assert atc.index.equals(targets.index)

### Identify features correlated with doubling rate

In [3]:
def important_features(train, dt, thres):

  # initialize dictionary to hold correlation results
  corr_dict = {}

  # correlate exp of each gene to drug response
  for feature in train.columns:
    corr_dict[feature] = train[feature].corr(dt)
  correlations = pd.DataFrame.from_dict(corr_dict, orient='index', columns=['Correlation'])

  # count number of univariable associations that meet the threshold
  num_pos = (correlations['Correlation'] > thres).sum()
  num_neg = (correlations['Correlation'] < -thres).sum()

  print('Selected threshold:', thres)
  print('Num features with positive correlation > threshold:', num_pos)
  print('Num features with |negative correlation| > threshold:', num_neg)
  print('Total number of features:', str(num_pos + num_neg))

  # identify features that pass selected threshold
  keep = correlations[correlations['Correlation'].abs() > thres].index

  # subset training dataframe to only genes of interest
  X_subset = train[keep]

  return X_subset

In [4]:
atc = important_features(atc, dt, 0.3)

Selected threshold: 0.3
Num features with positive correlation > threshold: 1815
Num features with |negative correlation| > threshold: 1938
Total number of features: 3753


In [5]:
rna = important_features(rna, dt, 0.25)

Selected threshold: 0.25
Num features with positive correlation > threshold: 733
Num features with |negative correlation| > threshold: 1571
Total number of features: 2304


In [6]:
cnv = important_features(cnv, dt, 0.2)

Selected threshold: 0.2
Num features with positive correlation > threshold: 2433
Num features with |negative correlation| > threshold: 496
Total number of features: 2929


In [7]:
mut = important_features(mut, dt, 0.15)

Selected threshold: 0.15
Num features with positive correlation > threshold: 1088
Num features with |negative correlation| > threshold: 251
Total number of features: 1339


### Identify correlated features

In [10]:
def corr_features(X_subset, thres):

    # correlate exp of remaining genes
    corr_mat = X_subset.corr(method='pearson', min_periods=1)
    vals = corr_mat.values.copy()
    np.fill_diagonal(vals, 0)
    corr_mat.iloc[:, :] = vals  

    corr_pairs = (corr_mat.abs() > thres)
    correlated = set() # initialize set to store correlated features

    # loop through correlated pairs
    for i in range(corr_pairs.shape[0]):
        for j in range(i + 1, corr_pairs.shape[1]):

            # if True (highly correlated)
            if corr_pairs.iloc[i, j]:

                #print(corr_mat.columns[i], corr_mat.columns[j])

                # add one of the genes to the set
                correlated.add(corr_mat.columns[i])

    #print('Highly correlated features:', correlated)
    print('Num correlated features:', len(correlated))

    print('Original number of feaures:', X_subset.shape[1])

    # remove correlated genes
    X_subset = X_subset.drop(columns=list(correlated))
    print('Number of features remaining:', X_subset.shape[1])

    return X_subset

In [11]:
atc = corr_features(atc, 0.8)

Num correlated features: 351
Original number of feaures: 3753
Number of features remaining: 3402


In [12]:
rna = corr_features(rna, 0.8)

Num correlated features: 795
Original number of feaures: 2304
Number of features remaining: 1509


In [13]:
cnv = corr_features(cnv, 0.8)

Num correlated features: 2885
Original number of feaures: 2929
Number of features remaining: 44


In [14]:
mut = corr_features(mut, 0.8)

Num correlated features: 882
Original number of feaures: 1339
Number of features remaining: 457


### Save new files

In [15]:
atc.to_csv("data/procdata/model_inputs/atc.csv", index=True)
rna.to_csv("data/procdata/model_inputs/rna.csv", index=True)
cnv.to_csv("data/procdata/model_inputs/cnv.csv", index=True)
mut.to_csv("data/procdata/model_inputs/mut.csv", index=True)
targets.to_csv("data/procdata/model_inputs/meta.csv", index=True)
